# Notebook Setup

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.stats as stats

# Portfolio Setup

In [2]:
tickers = ['AAPL', 'TSLA', 'MSFT', 'META', 'AMZN', 'GOOGL', 'NVDA'] #Ticker der MAG7 
start_capital = 100000

In [3]:
data = yf.download(tickers, start="2016-04-01", end="2026-04-01", auto_adjust=False)['Close']

[*********************100%***********************]  7 of 7 completed


In [4]:
benchmark_world_data = yf.download("VT", start="2016-04-01", end="2026-04-01", auto_adjust=False)['Close'] #VT (Vanguard Total World Stock ETF) 
benchmark_risk_free_data = yf.download("^TNX", start="2016-04-01", end="2026-04-01", auto_adjust=False)['Close'] #^TNX (10-Year Treasury Note Yield)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
#Berechnung der täglichen Renditen (diskret und logarithmisch)
returns_discrete = data.pct_change().dropna()
returns_log = np.log(data / data.shift(1)).dropna()

#Portfolio-Gewichtung
weights = np.array([1/len(tickers)] * len(tickers)) #1/7 Gewichtung für jedes Asset

#Berechnung der Portfolio-Renditen
portfolio_returns_discrete = returns_discrete.dot(weights)
portfolio_returns_log = returns_log.dot(weights)

Berechnung der täglichen relativen Renditen macht es möglich, dass Daten einer Normalverteilung angenähert werden können. Werden benötigt,um Erwartungswert und Volatilität zu brechnen

In [6]:
portfolio_returns_discrete.tail()

Date
2026-03-25    0.007628
2026-03-26   -0.031970
2026-03-27   -0.027635
2026-03-30   -0.001346
2026-03-31    0.045284
dtype: float64

# Erstellung der Funktionen

## Brechnung 1-Tages-Risikomodelle 

Value at Risk und expected Shortfall für Gaussian, lognorm und historische Modelle

Formel Gaussian Value at Risk: VaR = -(Erwartungswert(portfolio)+Z-Wert(alpha)*Volatilität)*Kapital 

In [7]:
# 1. Historisches Risiko (VaR & ES)
def calculate_historical_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()
    
    # VaR in % berechnen
    var_pct = np.percentile(clean_returns, alpha * 100)
    
    # ES berechnen (nutzt den exakten var_pct von oben als Filter)
    tail_returns = clean_returns[clean_returns <= var_pct]
    es_pct = np.mean(tail_returns)
    
    # Gibt beide Werte gleichzeitig zurück
    return capital * abs(var_pct), capital * abs(es_pct) #eigentlich negativer Wert, annahme, dass Funktion von VaR und ES bekannt ist


# 2. Gaußsches Risiko (VaR & ES)
def calculate_gaussian_risk_1d(returns, capital, var_level):
    alpha = var_level
    clean_returns = returns.dropna()
    
    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns) #Standardabweichung der Renditen
    z_score = stats.norm.ppf(alpha) #Z-Wert 
    
    # VaR in % berechnen
    var_pct = mu + (z_score * sigma)
    
    # ES berechnen
    es_pct = mu - sigma * (stats.norm.pdf(z_score) / alpha)
    
    return capital * abs(var_pct), capital * abs(es_pct)


# 3. Lognormales Risiko (VaR & ES)
def calculate_lognormal_risk_1d(log_returns, capital, var_level):
    alpha = var_level
    clean_returns = log_returns.dropna()
    
    mu_log = np.mean(clean_returns)
    sigma_log = np.std(clean_returns)
    z_score = stats.norm.ppf(alpha)
    
    # VaR in % berechnen
    log_worst_case = mu_log + (z_score * sigma_log)
    var_pct = np.exp(log_worst_case) - 1 #Umwandlung mit exp, da log
    
    # ES berechnen
    expected_value_lognorm = np.exp(mu_log + (sigma_log**2) / 2) #Erwartungswert der Lognormalverteilung
    tail_factor = stats.norm.cdf(z_score - sigma_log) / alpha #Berechnet den Flächenanteil des Expected Shortfalls im logarithmischen "Tail" (Extrembereich)
    es_pct = (expected_value_lognorm * tail_factor) - 1
    
    return capital * abs(var_pct), capital * abs(es_pct)

## Skalierung auf Zeiträume: 1 Jahr, 5 Jahre und 10 Jahre 

In [8]:
def calculate_historical_risk(log_returns, capital, var_level, days):
    if days == 1:
        rolling_discrete = np.exp(log_returns.dropna()) - 1 #Kein Window, da nur 1 Tag betrachtet wird, auch keine Log-Returns, da diskrete Renditen für 1 Tag ausreichend
    else:
        rolling_log = log_returns.rolling(window=days).sum().dropna() #Rolling Window über die Log-Returns, da für mehrere Tage die Log-Returns additiv sind (Renditen werden multipliziert, Logarithmus macht sie additiv)
        
        if len(rolling_log) < 100: #Sicherheitscheck, um zu verhindern, dass zu wenige Datenpunkte für die Berechnung genutzt werden
            return np.nan, np.nan
            
        rolling_discrete = np.exp(rolling_log) - 1 #Umwandlung der rollenden Log-Returns in diskrete Renditen

    var_pct = np.percentile(rolling_discrete, var_level * 100)  #VaR in % berechnen mit 5. Perzentil der rollenden diskreten Renditen
    tail_returns = rolling_discrete[rolling_discrete <= var_pct] #Filtert die Renditen, die im "Tail" liegen (also kleiner oder gleich dem berechneten VaR)
    es_pct = np.mean(tail_returns) #ES berechnen als Durchschnitt der Renditen im Tail (also der Renditen, die den VaR überschreiten)

    return capital * var_pct, capital * es_pct #Berechnung der abosluten VaR und ES

In [9]:
def calculate_gaussian_risk(returns, capital, var_level, days):
    clean_returns = returns.dropna()
    
    mu_1d = np.mean(clean_returns) #Erwartungswert der Renditen Tagesbasis
    sigma_1d = np.std(clean_returns) #Volatilität der Renditen Tagesbasis
    
    mu_nd = mu_1d * days #Erwarungswert für n Tage (Renditen sind addititv)
    sigma_nd = sigma_1d * np.sqrt(days) #Volatilität für n Tage mal Wurzerl aus T (hier Anzahl der Tage)
    
    z_score = stats.norm.ppf(var_level) #Z-Wert für Normalverteilung
    
    var_pct = mu_nd + (z_score * sigma_nd) 
    es_pct = mu_nd - sigma_nd * (stats.norm.pdf(z_score) / var_level)
    
    return capital * var_pct, capital * es_pct

In [10]:
def calculate_lognormal_risk(log_returns, capital, var_level, days):
    clean_returns = log_returns.dropna()
    
    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)
    
    mu_log_nd = mu_log_1d * days
    sigma_log_nd = sigma_log_1d * np.sqrt(days)
    
    z_score = stats.norm.ppf(var_level)
    
    log_worst_case = mu_log_nd + (z_score * sigma_log_nd)
    var_pct = np.exp(log_worst_case) - 1 #Umwandlung mit exp, da log
    
    expected_value_lognorm = np.exp(mu_log_nd + (sigma_log_nd**2) / 2) #Erwartungswert der Lognormalverteilung
    tail_factor = stats.norm.cdf(z_score - sigma_log_nd) / var_level
    es_pct = (expected_value_lognorm * tail_factor) - 1
    
    return capital * var_pct, capital * es_pct

Monte-Carlo-Simulation:für langfristige Prognosen, weil sie als einziges Modell den realen Zinseszinseffekt über die Jahre korrekt abbildet und durch ihre mathematische Struktur (Lognormalverteilung) garantiert, dass der Wert deines Aktienportfolios niemals unrealistischerweise unter null Dollar fallen kann.

In [11]:
def calculate_monte_carlo_risk(log_returns, capital, var_level, days, simulations=10000,black_swan=False,inflation_rate=0.00): 
    clean_returns = log_returns.dropna()
    
    mu_log_1d = np.mean(clean_returns)
    sigma_log_1d = np.std(clean_returns)
    
    np.random.seed(2) #Für Reproduzierbarkeit der Ergebnisse

    # np.random.normal generiert ein Raster der Größe (Anzahl Tage) x (Anzahl Simulationen)
    simulated_daily_returns = np.random.normal(mu_log_1d, sigma_log_1d, (days, simulations))
    
    cumulative_log_returns = np.sum(simulated_daily_returns, axis=0) #kumuliert die täglichen Log-Renditen über die Anzahl der Tage, um die Gesamtrendite pro Simulationspfad zu erhalten (additiv, da Log-Renditen)
    
    final_values = capital * np.exp(cumulative_log_returns) #Die 10.000 finalen Portfoliowerte nach T Tagen berechnen (Lognormal)
    
    if inflation_rate > 0.0:
        years = days / 252 #Umrechnung der Tage in Jahre
        discount_factor = (1 + inflation_rate) ** years # Abzinsung der gesamten finalen Matrix auf heutige Kaufkraft
        final_values = final_values / discount_factor

    if black_swan:
        final_values = final_values * 0.5 #Simuliert einen Black Swan Crash, der Zeitpunkt bei fixen Startkapital egal

    var_end_value = np.percentile(final_values, var_level * 100) # VAR-Schwellenwert finden (Das schlechteste x % der finalen Werte)
    
    tail_values = final_values[final_values <= var_end_value] #AVG der Werte, die im "Tail" liegen (also kleiner oder gleich dem berechneten VaR-Endwert)
    es_end_value = np.mean(tail_values)
    
    # PnL (Profit and Loss) berechnen = Endwert minus Startkapital
    # Ergebnis positiv = echter Gewinn --> Ergebnis negativ = echter Verlust
    var_pnl = var_end_value - capital
    es_pnl = es_end_value - capital
    
    return var_pnl, es_pnl


Simuliert zukünftige Portfoliowerte durch Bootstrapping (Ziehen mit Zurücklegen aus der Historie).    Fängt echte, ungeschönte Marktschocks ein, ohne Normalverteilungs-Annahmen zu treffen.

In [12]:
def calculate_bootstrap_risk(log_returns, capital, var_level, days, simulations=10000, black_swan=False,inflation_rate=0.00):
    
    clean_returns = log_returns.dropna()
    
    np.random.seed(1) #Für Reproduzierbarkeit der Ergebnisse
    
    # 1. Der "Lostopf": Wir ziehen 10.000 Pfade mal 'days' Renditen aus der echten Historie.
    # replace=True ist das magische "Zurücklegen" in den Topf.
    simulated_daily_returns = np.random.choice(clean_returns, size=(days, simulations), replace=True)
    
    # 2. Tägliche Renditen aufsummieren
    cumulative_log_returns = np.sum(simulated_daily_returns, axis=0)
    
    # 3. Finalen Portfoliowerte nach T Tagen berechnen
    final_values = capital * np.exp(cumulative_log_returns)
    
    if inflation_rate > 0.0:
        years = days / 252 #Umrechnung der Tage in Jahre
        discount_factor = (1 + inflation_rate) ** years # Abzinsung der gesamten finalen Matrix auf heutige Kaufkraft
        final_values = final_values / discount_factor

    if black_swan:
        final_values = final_values * 0.5 #Simuliert einen Black Swan Crash, der Zeitpunkt bei fixen Startkapital egal

    # 4. Den VaR-Schwellenwert finden (Das schlechteste x % der finalen Werte)
    var_end_value = np.percentile(final_values, var_level * 100)
    
    # 5. Expected Shortfall berechnen (Durchschnitt der Werte, die NOCH schlechter sind)
    tail_values = final_values[final_values <= var_end_value]
    es_end_value = np.mean(tail_values)
    
    # 6. PnL (Profit and Loss) berechnen = Endwert minus Startkapital
    # Ergebnis positiv = Gewinn / Ergebnis negativ = Verlust
    var_pnl = var_end_value - capital
    es_pnl = es_end_value - capital
    
    return var_pnl, es_pnl

## Funktionen zu den Performance Metriken

Beta, Sharpe-Ratio, RSF und Treynor

In [13]:
def calculate_performance_kpis(portfolio_returns, benchmark_world_data, benchmark_risk_free_data):
    
    # Aufbereitung der Daten für die Berechnung der KPIs 
    benchmark_returns = benchmark_world_data.pct_change().dropna().squeeze() #.squeeze() "quetscht" die DataFrame-Struktur, damit wir eine Series haben
    risk_free_daily = (benchmark_risk_free_data.dropna() / 100 / 252).squeeze() #/100 um auf Prozente zu kommen 

    # Daten synchronisieren (gleiche Handelstage usw. hier als für flexibilität bei in USA gehandelten Assets gleich)
    aligned_kpi_data = pd.concat([
        portfolio_returns.squeeze().rename('Portfolio'), 
        benchmark_returns.rename('Market'), 
        risk_free_daily.rename('RiskFree')
    ], axis=1, sort=False).dropna()

    port_ret = aligned_kpi_data['Portfolio'] #Portfolio-Renditen (hier Returns, also ret)
    mkt_ret = aligned_kpi_data['Market']
    rf_ret = aligned_kpi_data['RiskFree']

    #Berechnung der KPIs (Tagesbasis und Annualisiert)

    # Beta berechnen (Kovarianz Portfolio&Markt / Varianz Markt)
    cov_matrix = np.cov(port_ret, mkt_ret)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] #Beta als Maß wie stark Portfolio im Vergleich zum Markt schwankt 

    # Erwartungswerte (Durchschnitte)
    mu_rp_daily = np.mean(port_ret) #Erwartungswert der Portfolio-Renditen
    mu_rm_daily = np.mean(mkt_ret) #Erwartungswert der Markt-Renditen
    mu_rf_daily = np.mean(rf_ret) #Erwartungswert der risikofreien Renditen
    sigma_rp_daily = np.std(port_ret) #Standardabweichung der Portfolio-Renditen

    # Sharpe Ratio
    sharpe_daily = (mu_rp_daily - mu_rf_daily) / sigma_rp_daily
    sharpe_ann = sharpe_daily * np.sqrt(252)

    # Roy's Safety First Ratio (Nutzung vermeitlich Risikofreier Zins, fraglich ob USA-Staatsanleihen als risikofreier Zins gilt)
    rsf_daily = (mu_rp_daily - mu_rm_daily) / sigma_rp_daily
    rsf_ann = rsf_daily * np.sqrt(252)

    # Treynor Ratio (direkt Jahr dan Beta annualisiert ist)
    mu_rp_anno = mu_rp_daily * 252
    mu_rf_anno = mu_rf_daily * 252
    treynor_ann = (mu_rp_anno - mu_rf_anno) / beta

    # --- 3. Ausgabe ---
    print(f"\n{' PERFORMANCE KPIS (Annualisiert) ':=^75}")
    print(f"Portfolio Beta:      {beta:.2f}")
    print(f"Sharpe Ratio:        {sharpe_ann:.2f}")
    print(f"Roy's Safety First:  {rsf_ann:.2f}")
    print(f"Treynor Ratio:       {treynor_ann:.4f}")
    
    # Rückgabe der berechneten Werte als Dictionary für die spätere Weiterverwendung
    return {
        "Beta": beta,
        "Sharpe_Ratio": sharpe_ann,
        "Roys_Safety_First": rsf_ann,
        "Treynor_Ratio": treynor_ann
    }


## Funktionen Sparplanmodus mit Black Swan Modus und Inflation

In [14]:
def calculate_savings_plan_mc(log_returns, monthly_savings, years, simulations=10000,black_swan=False,inflation_rate=0.00):

    days = years * 252 #Anzahl der Handelstage über die gesamte Laufzeit
    clean_returns = log_returns.dropna()
    
    # Parameter für die Simulation
    mu = np.mean(clean_returns) #Erwartungswert der Renditen
    sigma = np.std(clean_returns)  #Volatilität der Renditen
    
    np.random.seed(2) # Für Reproduzierbarkeit
    
    daily_log_returns = np.random.normal(mu, sigma, (days, simulations)) #Generiert eine Matrix der täglichen Log-Renditen für alle Tage und Simulationen
    daily_simple_returns = np.exp(daily_log_returns) #Umwandlung der Log-Renditen
    
    #Hier entsteht die Black Swan Logik:

    if black_swan:
        crash_days = np.random.randint(0, days, size=simulations) #Wählt zufällig einen Tag für jeden Simulationspfad aus, an dem der Crash stattfindet
        
        for sim_idx in range(simulations): #In der Funktion wird sichergestellt, das jede Savings-Simulation später einen Crash-Tag hat
            crash_day = crash_days[sim_idx]
            daily_simple_returns[crash_day, sim_idx] *= 0.5 #Simuliert einen Crash an dem Portfoliowert sich halbiert


    #Hier entsteht die Sparplan-Logik:

    savings_matrix = np.zeros((days, simulations)) 
    for d in range(0, days, 21): #Annahme 252 Tage pro Jahr, also alle 21 Tage eine Einzahlung (252/12 = 21)
        savings_matrix[d, :] = monthly_savings #Matrix, die alle 21 Tage die Einzahlung enthält, ansonsten 0
    
    portfolio_values = np.zeros((days, simulations)) #Matrix, die die Portfoliowerte über die Zeit für alle Simulationen speichert
    current_values = np.zeros(simulations) #Startet bei 0, da wir mit einem Sparplan beginnen, kein Anfangskapital, speichert den aktuellen Portfoliowert von Tag zu Tag für jede Simulation
    
    for d in range(days):
        current_values = current_values * daily_simple_returns[d, :] + savings_matrix[d, :] # Rediten werden täglich aufsummiert, gleichzeitig werden alle 21 Tage eine Einzahlung hinzugefügt
        portfolio_values[d, :] = current_values #Speichert den Portfoliowert am Ende jedes Tages
        
    final_values = portfolio_values[-1, :] #Endwert der 10.000 Simulationen nach T Tagen
    
    #Inflationslogik:
    if inflation_rate > 0.0:
        discount_factor = (1 + inflation_rate) ** years # Abzinsung auf heutige Kaufkraft (Exponent sind die Jahre)
        final_values = final_values / discount_factor

    # VaR und ES berechnen (wie gehabt)
    # Aber Achtung: Hier vergleichen wir den Endwert mit der Summe der Einzahlungen!
    total_invested = (days // 21) * monthly_savings
    
    var_end_value = np.percentile(final_values, 5) # VAR-Schwellenwert finden (Das schlechteste 5 % der finalen Werte)
    es_end_value = np.mean(final_values[final_values <= var_end_value]) #AVG der Werte im 5% Tail
    
    # Profit/Loss im Vergleich zur Einzahlungssumme
    var_pnl = var_end_value - total_invested
    es_pnl = es_end_value - total_invested
    
    return var_pnl, es_pnl, final_values, total_invested

# Berechnung und Ausführung der Funktionen

## 1-Tages-Modelle

In [15]:
# var_level (alpha) festlegen
alpha = 0.05 

hist_var, hist_es = calculate_historical_risk_1d(portfolio_returns_discrete, start_capital, alpha)
gauss_var, gauss_es = calculate_gaussian_risk_1d(portfolio_returns_discrete, start_capital, alpha)
lognorm_var, lognorm_es = calculate_lognormal_risk_1d(portfolio_returns_log, start_capital, alpha)

# Ausgabe
print(f"Historisch: VaR = {hist_var:,.2f} $, ES = {hist_es:,.2f} $")
print(f"Gaußsch:    VaR = {gauss_var:,.2f} $, ES = {gauss_es:,.2f} $")
print(f"Lognormal:  VaR = {lognorm_var:,.2f} $, ES = {lognorm_es:,.2f} $")

Historisch: VaR = 3,000.60 $, ES = 4,251.25 $
Gaußsch:    VaR = 2,855.45 $, ES = 3,614.81 $
Lognormal:  VaR = 2,847.90 $, ES = 3,581.36 $


## Skalierung auf 1 Jahr, 5 & 10 Jahre

In [16]:
alpha_level = 0.05
inflation_rate = 0.02

horizons = {
    "1 Jahr": 252,
    "5 Jahre": 1260,
    "10 Jahre": 2520
}

print(f"\n{' RISIKOANALYSE (VaR & ES) ':=^80}")

for label, days in horizons.items():
    print(f"\n--- HORIZONT: {label} ({days} Handelstage) ---")
    print(f"{'Modell':<30} | {'Value at Risk':>15} | {'Expected Shortfall':>20}")
    print("-" * 72)
    
    if days == 252:
        # Für 1 Jahr: Die klassischen, analytischen und historischen Modelle
        h_var, h_es = calculate_historical_risk(portfolio_returns_log, start_capital, alpha_level, days)
        g_var, g_es = calculate_gaussian_risk(portfolio_returns_discrete, start_capital, alpha_level, days)
        l_var, l_es = calculate_lognormal_risk(portfolio_returns_log, start_capital, alpha_level, days)
        
        print(f"{'Historisch (BHS)':<30} | {h_var:>13,.2f} $ | {h_es:>18,.2f} $")
        print(f"{'Gaußsch':<30} | {g_var:>13,.2f} $ | {g_es:>18,.2f} $")
        print(f"{'Lognormal':<30} | {l_var:>13,.2f} $ | {l_es:>18,.2f} $")
        
    else:
        # Für 5 und 10 Jahre
        mc_var, mc_es = calculate_monte_carlo_risk(portfolio_returns_log, start_capital, alpha_level, days,black_swan=False,inflation_rate=inflation_rate)
        boot_var, boot_es = calculate_bootstrap_risk(portfolio_returns_log, start_capital, alpha_level, days,black_swan=False,inflation_rate=inflation_rate)
        
        print(f"{'Lognormal (Monte Carlo)':<30} | {mc_var:>13,.2f} $ | {mc_es:>18,.2f} $")
        print(f"{'Empirisch (Bootstrapping)':<30} | {boot_var:>13,.2f} $ | {boot_es:>18,.2f} $")

print("=" * 80)


=========================== RISIKOANALYSE (VaR & ES) ===========================

--- HORIZONT: 1 Jahr (252 Handelstage) ---
Modell                         |   Value at Risk |   Expected Shortfall
------------------------------------------------------------------------
Historisch (BHS)               |    -23,066.50 $ |         -36,000.45 $
Gaußsch                        |    -13,764.82 $ |         -25,819.27 $
Lognormal                      |    -19,419.64 $ |         -28,187.29 $

--- HORIZONT: 5 Jahre (1260 Handelstage) ---
Modell                         |   Value at Risk |   Expected Shortfall
------------------------------------------------------------------------
Lognormal (Monte Carlo)        |     13,210.36 $ |         -12,690.35 $
Empirisch (Bootstrapping)      |     15,290.77 $ |         -10,428.06 $

--- HORIZONT: 10 Jahre (2520 Handelstage) ---
Modell                         |   Value at Risk |   Expected Shortfall
-----------------------------------------------------------

## Performance KPIs 

In [17]:
kpi_results = calculate_performance_kpis(
    portfolio_returns=portfolio_returns_discrete, 
    benchmark_world_data=benchmark_world_data, 
    benchmark_risk_free_data=benchmark_risk_free_data
)


===================== PERFORMANCE KPIS (Annualisiert) =====================
Portfolio Beta:      1.34
Sharpe Ratio:        1.06
Roy's Safety First:  0.80
Treynor Ratio:       0.2275


Kurze Interpretation der KPIs:
Beta: Protfolio ist riskanter als Gesamtmarkt, schwankt 35% mehr als der Weltmarkt --> Steigt Weltmarkt um 10% steigt MAG7 um 13.5%. Sinkt Weltmarkt um 10% sinkt MAG7 um 13.5%. 

Sharpe Ratio: Für jede Einheit Risiko gibt es 1.07 Einheiten Überrendite im Vergleich zur riskfree Benchmark 

Roys SF: Für jede Einheit Risiko kann Portfolio 0.73% mehr Rednite erwirtschaften als Marktportfolio

Treynor Ratio: Für jede Einheit Marktrisiko erwirtschaftet MAG7 22.89% mehr Rendite im Vergleich mit dem Marktportfolio 

## Sparplanfunktion 

In [18]:
monatliche_rate = 100 
jahre_laufzeit = 20
inflation_rate = 0.02 #2% Inflation pro Jahr


# Funktionsaufruf
var_pnl, es_pnl, final_values, total_sum = calculate_savings_plan_mc(
    portfolio_returns_log, 
    monatliche_rate, 
    jahre_laufzeit, black_swan=True,inflation_rate=inflation_rate
)

# Schicke Formatierung der Ergebnisse
print(f"\n{' SPARPLAN ANALYSE (Monte Carlo) ':=^80}")
print(f"Zeitraum:            {jahre_laufzeit} Jahre")
print(f"Monatliche Rate:     {monatliche_rate:>10,.2f} $")
print(f"Gesamt investiert:   {total_sum:>10,.2f} $")
print("-" * 80)

# Interpretation der Ergebnisse
status_var = "GEWINN" if var_pnl > 0 else "VERLUST"
status_es = "GEWINN" if es_pnl > 0 else "VERLUST"

print(f"{'Value at Risk (95%):':<25} {abs(var_pnl):>12,.2f} $ {status_var}")
print(f"{'Expected Shortfall:':<25} {abs(es_pnl):>12,.2f} $ {status_es}")
print("-" * 80)
print(f"Durchschnittlicher Endwert: {np.mean(final_values):>12,.2f} $")
print("=" * 80)


======================== SPARPLAN ANALYSE (Monte Carlo) ========================
Zeitraum:            20 Jahre
Monatliche Rate:         100.00 $
Gesamt investiert:    24,000.00 $
--------------------------------------------------------------------------------
Value at Risk (95%):         38,998.06 $ GEWINN
Expected Shortfall:          21,163.91 $ GEWINN
--------------------------------------------------------------------------------
Durchschnittlicher Endwert:   662,362.79 $
